In [1]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random

# from pomelo.utils import Hal

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/psycopg2/__init__.py:144: UserWarning: The psycopg2 wheel package will be renamed from release 2.8; in order to keep installing from binary please use "pip install psycopg2-binary" instead. For details see: <http://initd.org/psycopg/docs/install.html#binary-install-from-pypi>.
  """)


In [26]:
# if import error run this cell first
!pip install --upgrade google-api-python-client oauth2client
!pip install gspread

In [27]:
import boto3

s3 = boto3.resource("s3")

In [28]:
#### authenticate your identity using the JSON credentials and provide access to your Google Drive #####
from apiclient import discovery
from httplib2 import Http
import oauth2client
from oauth2client import file, client, tools

obj = lambda: None
lmao = {
    "auth_host_name": "localhost",
    "noauth_local_webserver": "store_true",
    "auth_host_port": [8080, 8090],
    "logging_level": "ERROR",
}
for k, v in lmao.items():
    setattr(obj, k, v)

# authorization boilerplate code
SCOPES = "https://www.googleapis.com/auth/drive.readonly"
store = file.Storage("token.json")
creds = store.get()
# creds = None
# The following will give you a link if token.json does not exist, the link allows the user to give this app permission
if not creds or creds.invalid:
    flow = client.flow_from_clientsecrets(
        "/home/ec2-user/SageMaker/business-intelligence-notebooks/dfm_clothing/online_dfm_v2/client_secret.json",
        SCOPES,
    )
    creds = tools.run_flow(flow, store, obj)

In [29]:
import io
from googleapiclient.http import MediaIoBaseDownload

DRIVE = discovery.build("drive", "v3", http=creds.authorize(Http()))
# https://docs.google.com/spreadsheets/d/1LKibyPJvNLSvPR4A5t3zcdEBPAxrFuy4/edit#gid=1796573573
file_id = "1LKibyPJvNLSvPR4A5t3zcdEBPAxrFuy4"
request = DRIVE.files().get_media(fileId=file_id)
# replace the filename and extension in the first field below
fh = io.FileIO("store_traffic.xlsx", mode="w")
downloader = MediaIoBaseDownload(fh, request)
done = False
while done is False:
    status, done = downloader.next_chunk()
    print("Download %d%%." % int(status.progress() * 100))

Download 100%.


In [30]:
df = pd.read_excel(
    "store_traffic.xlsx", engine="openpyxl", sheet_name="unpivot_forecast"
)

In [31]:
df

,Store_name,month,year,Store_traffic,id_store,oldcluster,newcluster,country,warehouse
0,Central World,September,2021,13340.235280,261.0,sc2,FF,TH,TH
1,Central World,October,2021,17311.608180,261.0,sc2,FF,TH,TH
2,Central World,November,2021,19616.983110,261.0,sc2,FF,TH,TH
3,Central World,December,2021,21746.492110,261.0,sc2,FF,TH,TH
4,Central World,January,2022,11086.300000,261.0,sc2,FF,TH,TH
...,...,...,...,...,...,...,...,...,...
163,MY 1 Utama,November,2021,4918.500691,111.0,sc2,FFF,MY,MY
164,MY 1 Utama,December,2021,5682.233856,111.0,sc2,FFF,MY,MY
165,MY 1 Utama,January,2022,7176.000000,111.0,sc2,FFF,MY,MY
166,MY 1 Utama,February,2022,8949.000000,111.0,sc2,FFF,MY,MY


In [32]:
df.rename(columns={"Store_traffic": "tot_traffic"}, inplace=True)

In [33]:
df["tot_traffic"] = df["tot_traffic"].astype(np.int64)

In [34]:
df

,Store_name,month,year,tot_traffic,id_store,oldcluster,newcluster,country,warehouse
0,Central World,September,2021,13340,261.0,sc2,FF,TH,TH
1,Central World,October,2021,17311,261.0,sc2,FF,TH,TH
2,Central World,November,2021,19616,261.0,sc2,FF,TH,TH
3,Central World,December,2021,21746,261.0,sc2,FF,TH,TH
4,Central World,January,2022,11086,261.0,sc2,FF,TH,TH
...,...,...,...,...,...,...,...,...,...
163,MY 1 Utama,November,2021,4918,111.0,sc2,FFF,MY,MY
164,MY 1 Utama,December,2021,5682,111.0,sc2,FFF,MY,MY
165,MY 1 Utama,January,2022,7176,111.0,sc2,FFF,MY,MY
166,MY 1 Utama,February,2022,8949,111.0,sc2,FFF,MY,MY


In [36]:
store_traffic = (
    df.groupby(["year", "month", "oldcluster", "country"])["tot_traffic"]
    .mean()
    .reset_index()
)

In [38]:
store_traffic.rename(
    columns={"oldcluster": "cluster", "country": "id_shop_name"}, inplace=True
)

In [41]:
store_traffic

,year,month,cluster,id_shop_name,tot_traffic
0,2021,December,sc1,SG,30727.000000
1,2021,December,sc1,TH,37709.500000
2,2021,December,sc2,MY,5682.000000
3,2021,December,sc2,TH,13908.714286
4,2021,December,sc3,ID,22657.000000
5,2021,December,sc3,SG,19110.500000
6,2021,December,sc3,TH,11388.555556
7,2021,November,sc1,SG,25338.000000
8,2021,November,sc1,TH,31718.500000
9,2021,November,sc2,MY,4918.000000


In [40]:
store_traffic.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/deploy_store_traffic.csv",
    index=False,
)